# Talking to ViT-Up

Open-vocabulary segmentation on high-resolution ViT-Up-B features.

This notebook uses [`lorebianchi98/Talk2DINOv3-ViTB`](https://huggingface.co/lorebianchi98/Talk2DINOv3-ViTB) to map a text prompt into DINOv3 ViT-B feature space, extracts dense ViT-Up-B features for an image from `assets/`, computes cosine similarity, visualizes PCA and heatmaps, and displays a thresholded segmentation mask.

Important: the threshold is applied to the Talk2DINO score `sigmoid(cosine_similarity)`. The min-max normalized cosine map is only for visualization, because raw cosine values are not naturally in `[0, 1]`.


In [ ]:
from pathlib import Path

import ipywidgets as widgets
import torch
from IPython.display import display

from vit_up.demo.talking_to_vitup_utils import (
    TalkingToVitUpRunner,
    get_device,
    get_repo_root,
    list_asset_images,
    load_talk2dino_model,
    load_vit_up_model,
    render_result,
)

repo_root = get_repo_root()
assets_dir = repo_root / "assets"
image_paths = list_asset_images(assets_dir)

device = get_device()
torch.set_grad_enabled(False)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
print(f"Assets: {[p.name for p in image_paths]}")


In [ ]:
vit_up_model = load_vit_up_model(device)

print("Loading Talk2DINO text encoder...")
talk2dino_model = load_talk2dino_model(device)

runner = TalkingToVitUpRunner(
    vit_up_model=vit_up_model,
    talk2dino_model=talk2dino_model,
    device=device,
)
print("Models ready.")


In [ ]:
default_image = assets_dir / "messy_room.png"

image_widget = widgets.Dropdown(
    options=[(p.name, str(p)) for p in image_paths],
    value=str(default_image) if default_image.exists() else str(image_paths[0]),
    description="Image",
    layout=widgets.Layout(width="420px"),
)
prompt_widget = widgets.Text(
    value="basket",
    description="Prompt",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
query_res_widget = widgets.IntSlider(
    value=512,
    min=224,
    max=1024,
    step=16,
    description="Max side",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)
threshold_mode_widget = widgets.Dropdown(
    options=[
        ("Talk2DINO bg 0.55", "talk2dino_bg"),
        ("Otsu", "otsu"),
        ("Percentile", "percentile"),
        ("Manual", "manual"),
    ],
    value="percentile",
    description="Score cutoff",
    layout=widgets.Layout(width="420px"),
)
threshold_widget = widgets.FloatSlider(
    value=0.55,
    min=0.0,
    max=1.0,
    step=0.01,
    description="Manual score",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="420px"),
)
percentile_widget = widgets.FloatSlider(
    value=97.5,
    min=50.0,
    max=99.5,
    step=0.5,
    description="Score pct",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="420px"),
)
refine_widget = widgets.Checkbox(
    value=False,
    description="PAMR refine",
    indent=False,
    layout=widgets.Layout(width="420px"),
)

controls = widgets.VBox([
    widgets.HBox([image_widget, prompt_widget]),
    widgets.HBox([query_res_widget, threshold_mode_widget]),
    widgets.HBox([threshold_widget, percentile_widget, refine_widget]),
])


def update_view(image_path, prompt, max_side, threshold_mode, threshold, percentile, refine):
    result = runner.run(
        image_path=image_path,
        prompt=prompt,
        max_side=max_side,
    )
    render_result(
        result,
        threshold_mode=threshold_mode,
        threshold=threshold,
        percentile=percentile,
        refine=refine,
    )

out = widgets.interactive_output(
    update_view,
    {
        "image_path": image_widget,
        "prompt": prompt_widget,
        "max_side": query_res_widget,
        "threshold_mode": threshold_mode_widget,
        "threshold": threshold_widget,
        "percentile": percentile_widget,
        "refine": refine_widget,
    },
)

display(widgets.VBox([controls, out]))
